In [ ]:
import openneuro

dm_path = '/home/manu/TFG2/ds003766_data'

openneuro.download(
    dataset='ds003766',
    target_dir=dm_path
)

In [10]:
import mne
import numpy as np
import pandas as pd
import os
import eegraph
from scipy import signal


In [11]:
base_path = '/home/manu/TFG2/ds003766_aws'
deriv_path = base_path + '/derivatives/eeglab-preproc'

subjects = [f'sub-{i:02d}' for i in range(1, 32)]
print(f"Sujetos: {len(subjects)}")

Sujetos: 31


In [12]:
# Usar los canales que realmente existen
available_channels = ['E2', 'E3', 'E4', 'E5', 'E6', 'E7', 'E9', 'E10', 'E11', 'E12', 'E13', 'E15', 'E16', 'E18', 'E19', 'E20', 'E22', 'E23', 'E24', 'E26', 'E27', 'E28', 'E29', 'E30', 'E31', 'E33', 'E34', 'E35', 'E36', 'E37', 'E39', 'E40', 'E41', 'E42', 'E44', 'E45', 'E46', 'E47', 'E50', 'E51', 'E52', 'E53', 'E54', 'E55', 'E57', 'E58', 'E59', 'E60', 'E61', 'E62', 'E64', 'E65', 'E66', 'E67', 'E69', 'E70', 'E71', 'E72', 'E74', 'E75', 'E76', 'E77', 'E78', 'E79', 'E80', 'E82', 'E83', 'E84', 'E85', 'E86', 'E87', 'E89', 'E90', 'E91', 'E92', 'E93', 'E95', 'E96', 'E97', 'E98', 'E100', 'E101', 'E102', 'E103', 'E104', 'E105', 'E106', 'E108', 'E109', 'E110', 'E111', 'E112', 'E114', 'E115', 'E116', 'E117', 'E118', 'E122', 'E123', 'E124', 'E129']

channels_64 = available_channels[:64]
channels_32 = available_channels[:64:2]  # Cada 2
channels_16 = available_channels[:64:4]  # Cada 4
channels_8 = available_channels[:64:8]   # Cada 8

print(f"64 canales: {len(channels_64)}")
print(f"32 canales: {len(channels_32)}")
print(f"16 canales: {len(channels_16)}")
print(f"8 canales: {len(channels_8)}")

64 canales: 64
32 canales: 32
16 canales: 16
8 canales: 8


In [13]:
output_base = '/home/manu/TFG2/EEG-Gold-Standard-main/ConnectivityMatrices_DS003766'

for split in ['Training', 'Test']:
    for ch in ['8', '16', '32', '64']:
        for label in ['left', 'right']:
            path = f'{output_base}/{split}/{ch}/{label}'
            os.makedirs(path, exist_ok=True)

print("Carpetas creadas")

Carpetas creadas


In [2]:
import os

output_base = '/home/manu/TFG2/EEG-Gold-Standard-main/ConnectivityMatrices_DS003766_v2'

for split in ['Training', 'Test']:
    for ch in ['8', '16', '32', '64']:
        for label in ['resting', 'task']:
            path = f'{output_base}/{split}/{ch}/{label}'
            os.makedirs(path, exist_ok=True)

print("Carpetas creadas")

Carpetas creadas


In [3]:
import shutil
import os

old_base = '/home/manu/TFG2/EEG-Gold-Standard-main/ConnectivityMatrices_DS003766'
new_base = '/home/manu/TFG2/EEG-Gold-Standard-main/ConnectivityMatrices_DS003766_v2'

for split in ['Training', 'Test']:
    for ch in ['8', '16', '32', '64']:
        # Copiar left y right a task
        for label in ['left', 'right']:
            src_folder = f'{old_base}/{split}/{ch}/{label}'
            dst_folder = f'{new_base}/{split}/{ch}/task'
            
            for file in os.listdir(src_folder):
                src = os.path.join(src_folder, file)
                dst = os.path.join(dst_folder, file)
                shutil.copy(src, dst)
        
        # Contar
        n_task = len(os.listdir(f'{new_base}/{split}/{ch}/task'))
        print(f"{split}/{ch}: {n_task} matrices task")

print("\n¡Copiado!")

Training/8: 7472 matrices task
Training/16: 7472 matrices task
Training/32: 7472 matrices task
Training/64: 7472 matrices task
Test/8: 1763 matrices task
Test/16: 1763 matrices task
Test/32: 1763 matrices task
Test/64: 1763 matrices task

¡Copiado!


In [ ]:
def calculate_coherence(raw_segment, fmin=8, fmax=12):
    """Calcula squared coherence en banda alpha usando scipy"""
    data = raw_segment.get_data()
    sfreq = raw_segment.info['sfreq']
    n_channels = data.shape[0]
    
    matrix = np.zeros((n_channels, n_channels))
    
    for i in range(n_channels):
        for j in range(i, n_channels):
            if i == j:
                matrix[i, j] = 1.0
            else:
                f, Cxy = signal.coherence(data[i], data[j], fs=sfreq, nperseg=50)
                alpha_mask = (f >= fmin) & (f <= fmax)
                coh_alpha = np.mean(Cxy[alpha_mask])
                matrix[i, j] = coh_alpha
                matrix[j, i] = coh_alpha
    
    return matrix

print("Función calculate_coherence definida")

In [9]:
from scipy import signal
import numpy as np
import mne

def calculate_coherence(data, sfreq, fmin=8, fmax=12):
    """Calcula squared coherence en banda alpha"""
    n_channels = data.shape[0]
    matrix = np.zeros((n_channels, n_channels))
    
    for i in range(n_channels):
        for j in range(i, n_channels):
            if i == j:
                matrix[i, j] = 1.0
            else:
                f, Cxy = signal.coherence(data[i], data[j], fs=sfreq, nperseg=50)
                alpha_mask = (f >= fmin) & (f <= fmax)
                coh_alpha = np.mean(Cxy[alpha_mask])
                matrix[i, j] = coh_alpha
                matrix[j, i] = coh_alpha
    return matrix

def process_resting(subject, deriv_path, output_base, channels, num_ch, split):
    """Procesa epochs de resting para un sujeto"""
    resting_file = f'{deriv_path}/{subject}/{subject}_task-resting_eeg.set'
    
    try:
        epochs = mne.io.read_epochs_eeglab(resting_file, verbose=False)
    except:
        return 0
    
    # Seleccionar canales
    epochs.pick_channels(channels, ordered=True)
    
    count = 0
    for i in range(len(epochs)):
        try:
            data = epochs[i].get_data()[0]  # (n_channels, n_times)
            matrix = calculate_coherence(data, epochs.info['sfreq'])
            
            filename = f'{output_base}/{split}/{num_ch}/resting/{subject}_resting_{i:03d}.npy'
            np.save(filename, matrix)
            count += 1
        except:
            continue
    
    return count

print("Función definida")

Función definida


In [6]:
def process_subject(subject, deriv_path, output_base, channels, num_ch, split):
    """Procesa un sujeto y genera matrices de conectividad"""
    
    eeg_file = f'{deriv_path}/{subject}/{subject}_task-foodchoice_eeg.set'
    raw = mne.io.read_raw_eeglab(eeg_file, preload=True, verbose=False)
    raw.pick_channels(channels, ordered=True)
    
    events_left = mne.events_from_annotations(raw, event_id={'0500': 1}, verbose=False)[0]
    events_right = mne.events_from_annotations(raw, event_id={'0501': 2}, verbose=False)[0]
    
    count = 0
    
    # Procesar LEFT
    for i, event in enumerate(events_left):
        onset_sec = event[0] / raw.info['sfreq']
        start = onset_sec - 1.0
        
        if start < 0:
            continue
        
        try:
            raw_segment = raw.copy().crop(tmin=start, tmax=onset_sec)
            matrix = calculate_coherence(raw_segment)
            filename = f'{output_base}/{split}/{num_ch}/left/{subject}_left_{i:03d}.npy'
            np.save(filename, matrix)
            count += 1
        except:
            continue
    
    # Procesar RIGHT
    for i, event in enumerate(events_right):
        onset_sec = event[0] / raw.info['sfreq']
        start = onset_sec - 1.0
        
        if start < 0:
            continue
        
        try:
            raw_segment = raw.copy().crop(tmin=start, tmax=onset_sec)
            matrix = calculate_coherence(raw_segment)
            filename = f'{output_base}/{split}/{num_ch}/right/{subject}_right_{i:03d}.npy'
            np.save(filename, matrix)
            count += 1
        except:
            continue
    
    return count

print("Función process_subject definida")

Función process_subject definida


In [10]:
deriv_path = '/home/manu/TFG2/ds003766_aws/derivatives/eeglab-preproc'
output_base = '/home/manu/TFG2/EEG-Gold-Standard-main/ConnectivityMatrices_DS003766_v2'

subjects = [f'sub-{i:02d}' for i in range(1, 32)]
train_subjects = subjects[:25]
test_subjects = subjects[25:] 
available_channels = ['E2', 'E3', 'E4', 'E5', 'E6', 'E7', 'E9', 'E10', 'E11', 'E12', 'E13', 'E15', 'E16', 'E18', 'E19', 'E20', 'E22', 'E23', 'E24', 'E26', 'E27', 'E28', 'E29', 'E30', 'E31', 'E33', 'E34', 'E35', 'E36', 'E37', 'E39', 'E40', 'E41', 'E42', 'E44', 'E45', 'E46', 'E47', 'E50', 'E51', 'E52', 'E53', 'E54', 'E55', 'E57', 'E58', 'E59', 'E60', 'E61', 'E62', 'E64', 'E65', 'E66', 'E67', 'E69', 'E70', 'E71', 'E72', 'E74', 'E75', 'E76', 'E77', 'E78', 'E79', 'E80', 'E82', 'E83', 'E84', 'E85', 'E86', 'E87', 'E89', 'E90', 'E91', 'E92', 'E93', 'E95', 'E96', 'E97', 'E98', 'E100', 'E101', 'E102', 'E103', 'E104', 'E105', 'E106', 'E108', 'E109', 'E110', 'E111', 'E112', 'E114', 'E115', 'E116', 'E117', 'E118', 'E122', 'E123', 'E124', 'E129']

channel_configs = {
    '64': available_channels[:64],
    '32': available_channels[:64:2],
    '16': available_channels[:64:4],
    '8': available_channels[:64:8]
}

# Training
print("=== TRAINING RESTING ===")
for num_ch, channels in channel_configs.items():
    print(f"\nProcesando {num_ch} canales...")
    total = 0
    for subject in train_subjects:
        count = process_resting(subject, deriv_path, output_base, channels, num_ch, 'Training')
        total += count
        print(f"  {subject}: {count} matrices")
    print(f"Total {num_ch} canales: {total}")

# Test
print("\n=== TEST RESTING ===")
for num_ch, channels in channel_configs.items():
    print(f"\nProcesando {num_ch} canales...")
    total = 0
    for subject in test_subjects:
        count = process_resting(subject, deriv_path, output_base, channels, num_ch, 'Test')
        total += count
        print(f"  {subject}: {count} matrices")
    print(f"Total {num_ch} canales: {total}")

print("\n¡Terminado!")

=== TRAINING RESTING ===

Procesando 64 canales...


/tmp/ipykernel_12549/568562678.py:27: RuntimeWarning: At least one epoch has multiple events. Only the latency of the first event will be retained.
  epochs = mne.io.read_epochs_eeglab(resting_file, verbose=False)


NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
  sub-01: 200 matrices


/tmp/ipykernel_12549/568562678.py:27: RuntimeWarning: At least one epoch has multiple events. Only the latency of the first event will be retained.
  epochs = mne.io.read_epochs_eeglab(resting_file, verbose=False)


NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
  sub-02: 200 matrices
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


...

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
  sub-29: 200 matrices
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_12549/568562678.py:27: RuntimeWarning: At least one epoch has multiple events. Only the latency of the first event will be retained.
  epochs = mne.io.read_epochs_eeglab(resting_file, verbose=False)


  sub-30: 200 matrices
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_12549/568562678.py:27: RuntimeWarning: At least one epoch has multiple events. Only the latency of the first event will be retained.
  epochs = mne.io.read_epochs_eeglab(resting_file, verbose=False)


  sub-31: 200 matrices
Total 8 canales: 1200

¡Terminado!


In [1]:
import os

output_base = '/home/manu/TFG2/EEG-Gold-Standard-main/ConnectivityMatrices_DS003766_v2'

for split in ['Training', 'Test']:
    print(f"=== {split} ===")
    for ch in ['64', '32', '16', '8']:
        resting = len(os.listdir(f'{output_base}/{split}/{ch}/resting'))
        task = len(os.listdir(f'{output_base}/{split}/{ch}/task'))
        print(f"  {ch} canales: resting={resting}, task={task}")
    print()

=== Training ===
  64 canales: resting=4996, task=7472
  32 canales: resting=4996, task=7472
  16 canales: resting=4996, task=7472
  8 canales: resting=4996, task=7472

=== Test ===
  64 canales: resting=1200, task=1763
  32 canales: resting=1200, task=1763
  16 canales: resting=1200, task=1763
  8 canales: resting=1200, task=1763



In [ ]:
train_subjects = subjects[:25] # sub-01 a sub-25
test_subjects = subjects[25:]  # sub-26 a sub-31

print(f"Training: {len(train_subjects)} sujetos")
print(f"Test: {len(test_subjects)} sujetos")

Training: 25 sujetos
Test: 6 sujetos


In [17]:
channel_configs = {
    '64': channels_64,
    '32': channels_32,
    '16': channels_16,
    '8': channels_8
}

# Training
print("=== TRAINING ===")
for num_ch, channels in channel_configs.items():
    print(f"\nProcesando {num_ch} canales...")
    total = 0
    for subject in train_subjects:
        count = process_subject(subject, deriv_path, output_base, channels, num_ch, 'Training')
        total += count
        print(f"  {subject}: {count} matrices")
    print(f"Total {num_ch} canales: {total}")

# Test
print("\n=== TEST ===")
for num_ch, channels in channel_configs.items():
    print(f"\nProcesando {num_ch} canales...")
    total = 0
    for subject in test_subjects:
        count = process_subject(subject, deriv_path, output_base, channels, num_ch, 'Test')
        total += count
        print(f"  {subject}: {count} matrices")
    print(f"Total {num_ch} canales: {total}")

print("\n¡Terminado!")

=== TRAINING ===

Procesando 64 canales...
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_20853/1909197477.py:5: RuntimeWarning: Limited 2 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_eeglab(eeg_file, preload=True, verbose=False)
/tmp/ipykernel_20853/1909197477.py:5: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(eeg_file, preload=True, verbose=False)


  sub-01: 297 matrices
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_20853/1909197477.py:5: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_eeglab(eeg_file, preload=True, verbose=False)
/tmp/ipykernel_20853/1909197477.py:5: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(eeg_file, preload=True, verbose=False)


  sub-02: 303 matrices
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


...

  sub-29: 289 matrices
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_20853/1909197477.py:5: RuntimeWarning: Limited 4 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_eeglab(eeg_file, preload=True, verbose=False)
/tmp/ipykernel_20853/1909197477.py:5: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(eeg_file, preload=True, verbose=False)


  sub-30: 307 matrices
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_20853/1909197477.py:5: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_eeglab(eeg_file, preload=True, verbose=False)
/tmp/ipykernel_20853/1909197477.py:5: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(eeg_file, preload=True, verbose=False)


  sub-31: 295 matrices
Total 8 canales: 1763

¡Terminado!


In [18]:
# Verificar distribución left/right
for split in ['Training', 'Test']:
    for ch in ['64', '32', '16', '8']:
        left_path = f'{output_base}/{split}/{ch}/left'
        right_path = f'{output_base}/{split}/{ch}/right'
        
        n_left = len(os.listdir(left_path))
        n_right = len(os.listdir(right_path))
        
        print(f"{split} {ch}ch: left={n_left}, right={n_right}")
    print()

Training 64ch: left=3735, right=3737
Training 32ch: left=3735, right=3737
Training 16ch: left=3735, right=3737
Training 8ch: left=3735, right=3737

Test 64ch: left=876, right=887
Test 32ch: left=876, right=887
Test 16ch: left=876, right=887
Test 8ch: left=876, right=887



In [19]:
from scipy import signal

def calculate_coherence(raw_segment, fmin=8, fmax=12):
    """Calcula squared coherence en banda alpha usando scipy"""
    data = raw_segment.get_data()
    sfreq = raw_segment.info['sfreq']
    n_channels = data.shape[0]
    
    matrix = np.zeros((n_channels, n_channels))
    
    for i in range(n_channels):
        for j in range(i, n_channels):
            if i == j:
                matrix[i, j] = 1.0
            else:
                f, Cxy = signal.coherence(data[i], data[j], fs=sfreq, nperseg=50)
                # Promediar en banda alpha (8-12 Hz)
                alpha_mask = (f >= fmin) & (f <= fmax)
                coh_alpha = np.mean(Cxy[alpha_mask])
                matrix[i, j] = coh_alpha
                matrix[j, i] = coh_alpha
    
    return matrix

# Probar
matrix = calculate_coherence(raw_segment)
print(f"Matriz shape: {matrix.shape}")
print(f"Valores min/max: {matrix.min():.3f} / {matrix.max():.3f}")
print(f"Media (sin diagonal): {matrix[np.triu_indices(64, k=1)].mean():.3f}")

Matriz shape: (64, 64)
Valores min/max: 0.075 / 1.000
Media (sin diagonal): 0.531


In [20]:
def process_subject(subject, deriv_path, output_base, channels, num_ch, split):
    """Procesa un sujeto y genera matrices de conectividad"""
    
    # 1. Cargar EEG
    eeg_file = f'{deriv_path}/{subject}/{subject}_task-foodchoice_eeg.set'
    raw = mne.io.read_raw_eeglab(eeg_file, preload=True, verbose=False)
    
    # 2. Seleccionar canales
    raw.pick_channels(channels, ordered=True)
    
    # 3. Obtener eventos (0500=left, 0501=right)
    events_left = mne.events_from_annotations(raw, event_id={'0500': 1}, verbose=False)[0]
    events_right = mne.events_from_annotations(raw, event_id={'0501': 2}, verbose=False)[0]
    
    count = 0
    
    # 4. Procesar eventos LEFT
    for i, event in enumerate(events_left):
        onset_sec = event[0] / raw.info['sfreq']
        start = onset_sec - 1.0
        end = onset_sec
        
        if start < 0:
            continue
            
        try:
            raw_segment = raw.copy().crop(tmin=start, tmax=end)
            matrix = calculate_coherence(raw_segment)
            
            filename = f'{output_base}/{split}/{num_ch}/left/{subject}_left_{i:03d}.npy'
            np.save(filename, matrix)
            count += 1
        except:
            continue
    
    # 5. Procesar eventos RIGHT
    for i, event in enumerate(events_right):
        onset_sec = event[0] / raw.info['sfreq']
        start = onset_sec - 1.0
        end = onset_sec
        
        if start < 0:
            continue
            
        try:
            raw_segment = raw.copy().crop(tmin=start, tmax=end)
            matrix = calculate_coherence(raw_segment)
            
            filename = f'{output_base}/{split}/{num_ch}/right/{subject}_right_{i:03d}.npy'
            np.save(filename, matrix)
            count += 1
        except:
            continue
    
    return count

print("Función actualizada")

Función actualizada


In [ ]:
channel_configs = {
    '64': channels_64,
    '32': channels_32,
    '16': channels_16,
    '8': channels_8
}

# Training
print("=== TRAINING ===")
for num_ch, channels in channel_configs.items():
    print(f"\nProcesando {num_ch} canales...")
    total = 0
    for subject in train_subjects:
        count = process_subject(subject, deriv_path, output_base, channels, num_ch, 'Training')
        total += count
        print(f"  {subject}: {count} matrices")
    print(f"Total {num_ch} canales: {total}")

# Test
print("\n=== TEST ===")
for num_ch, channels in channel_configs.items():
    print(f"\nProcesando {num_ch} canales...")
    total = 0
    for subject in test_subjects:
        count = process_subject(subject, deriv_path, output_base, channels, num_ch, 'Test')
        total += count
        print(f"  {subject}: {count} matrices")
    print(f"Total {num_ch} canales: {total}")

print("\n¡Terminado!")

=== TRAINING ===

Procesando 64 canales...


/tmp/ipykernel_15295/3715535392.py:6: RuntimeWarning: Limited 2 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_eeglab(eeg_file, preload=True, verbose=False)
/tmp/ipykernel_15295/3715535392.py:6: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(eeg_file, preload=True, verbose=False)
